In [4]:
!pip install pymoo matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 23.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 34.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [pymoo]


In [8]:
import numpy as np
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.sampling.rnd import IntegerRandomSampling
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.repair.rounding import RoundingRepair
from pymoo.optimize import minimize

class KubernetesClusterProblem(ElementwiseProblem):
    def __init__(self):
        # x1: Replicas da API (2 a 40), x2: Worker Nodes (1 a 12)
        super().__init__(
            n_var=2,
            n_obj=2,
            xl=np.array([2, 1]),
            xu=np.array([40, 12]),
            vtype=int
        )

    def _evaluate(self, x, out, *args, **kwargs):
        replicas, nodes = x[0], x[1]

        # TODO 1: Tempo de Resposta (ms)
        f1_tempo = (600.0 / replicas) + (200.0 / nodes)

        # TODO 2: Custo Mensal (USD)
        f2_custo = (replicas * 12.0) + (nodes * 85.0)

        out["F"] = [f1_tempo, f2_custo]

# TODO 3: NSGA2 com operadores inteiros e RoundingRepair
algorithm = NSGA2(
    pop_size=25,
    sampling=IntegerRandomSampling(),
    crossover=SBX(prob=0.9, eta=15, vtype=float, repair=RoundingRepair()),
    mutation=PM(prob=0.2, eta=20, vtype=float, repair=RoundingRepair()),
    eliminate_duplicates=True
)

# TODO 4: Minimizacao por 35 geracoes (seed=10)
res2 = minimize(KubernetesClusterProblem(), algorithm, ('n_gen', 35), seed=10)

print("Solucoes inteiras encontradas no Cluster Kubernetes (Replicas, Nodes):")
for sol_x, sol_f in zip(res2.X, res2.F):
    print(f"Replicas: {sol_x[0]:2d} | Nodes: {sol_x[1]:2d} => Tempo: {sol_f[0]:6.2f} ms | Custo: US$ {sol_f[1]:7.2f}")

Solucoes inteiras encontradas no Cluster Kubernetes (Replicas, Nodes):
Replicas:  2 | Nodes:  1 => Tempo: 500.00 ms | Custo: US$  109.00
Replicas: 40 | Nodes: 12 => Tempo:  31.67 ms | Custo: US$ 1500.00
Replicas:  3 | Nodes:  1 => Tempo: 400.00 ms | Custo: US$  121.00
Replicas:  4 | Nodes:  1 => Tempo: 350.00 ms | Custo: US$  133.00
Replicas: 33 | Nodes:  8 => Tempo:  43.18 ms | Custo: US$ 1076.00
Replicas: 39 | Nodes:  5 => Tempo:  55.38 ms | Custo: US$  893.00
Replicas:  7 | Nodes:  2 => Tempo: 185.71 ms | Custo: US$  254.00
Replicas:  5 | Nodes:  2 => Tempo: 220.00 ms | Custo: US$  230.00
Replicas: 16 | Nodes:  3 => Tempo: 104.17 ms | Custo: US$  447.00
Replicas: 10 | Nodes:  2 => Tempo: 160.00 ms | Custo: US$  290.00
Replicas: 11 | Nodes:  5 => Tempo:  94.55 ms | Custo: US$  557.00
Replicas: 18 | Nodes:  2 => Tempo: 133.33 ms | Custo: US$  386.00
Replicas: 13 | Nodes:  2 => Tempo: 146.15 ms | Custo: US$  326.00
Replicas:  5 | Nodes:  1 => Tempo: 320.00 ms | Custo: US$  145.00
Repli